In [9]:
import pandas as pd
import numpy as np
import keras
import tensorflow as tf
import shap
import matplotlib.pyplot as plt

seed=42

# 1. Data processing

In [10]:
df = pd.read_csv("val_data2.csv", sep=",")

### --- Normalize data
from sklearn.preprocessing import StandardScaler

features = ['halo_mass', 'd3', 'd5', 'd8', 'c_200', 'z', 'spin', 'd_min', 'd_node',
       'd_saddle_1', 'd_saddle_2', 'd_skel']
target = ['b1']

X = df[features]
y = df[target]

scaler = StandardScaler()
X_norm = scaler.fit_transform(X)

scaler = StandardScaler()
y_norm = scaler.fit_transform(y)

X_norm.shape, y_norm.shape

### --- train-val-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_norm,y_norm,test_size=0.2,random_state=seed)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)
X_train.shape, X_val.shape, X_test.shape

((9386, 12), (2347, 12), (2934, 12))

# 2. Model loading

In [3]:
### --- Download previously trained model
!gdown 'https://drive.google.com/file/d/1WT0bDVNxaHRCoFrz_DBXHOwtix6R4WKT/view?usp=sharing'

Downloading...
From: https://drive.google.com/uc?id=1WT0bDVNxaHRCoFrz_DBXHOwtix6R4WKT
To: /home/csori/semestre/012026/Proyecto-XAI/notebooks/GalaxyBiasResNet.keras
100%|██████████████████████████████████████| 7.16M/7.16M [00:01<00:00, 3.69MB/s]


In [7]:
### --- load model

model = keras.saving.load_model('GalaxyBiasResNet.keras')
#model.summary() #uncomment this line to see network layers

# 3. SHAP Explanations

In [ ]:
def predict_for_shap(x_2d):
    x_3d = tf.cast(np.asarray(x_2d).reshape((-1, X_train.shape[1], 1)), tf.float32)
    return model(x_3d, training=False).numpy().reshape(-1)

rng = np.random.default_rng(42)
background_idx = rng.choice(X_train.shape[0], size=500, replace=False)
explain_idx    = rng.choice(X_test.shape[0],  size=1800, replace=False)

X_background = X_train[background_idx]
X_explain    = X_test[explain_idx]

explainer   = shap.KernelExplainer(predict_for_shap, X_background)
shap_values = explainer.shap_values(X_explain, nsamples=120)

if isinstance(shap_values, list):
    shap_values = shap_values[0]

features = ['halo_mass', 'd3', 'd5', 'd8', 'c_200', 'z', 'spin',
            'd_min', 'd_node', 'd_saddle_1', 'd_saddle_2', 'd_skel']

Using 500 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.
  0%|          | 0/1800 [00:00<?, ?it/s]W0000 00:00:1779826421.530037 1273363 cpu_allocator_impl.cc:82] Allocation of 245760000 exceeds 10% of free system memory.
W0000 00:00:1779826421.797102 1273363 cpu_allocator_impl.cc:82] Allocation of 245760000 exceeds 10% of free system memory.
W0000 00:00:1779826421.972309 1273363 cpu_allocator_impl.cc:82] Allocation of 245760000 exceeds 10% of free system memory.
W0000 00:00:1779826422.043114 1273363 cpu_allocator_impl.cc:82] Allocation of 245760000 exceeds 10% of free system memory.
W0000 00:00:1779826422.164624 1273363 cpu_allocator_impl.cc:82] Allocation of 245760000 exceeds 10% of free system memory.


In [ ]:
# Global feature importance — beeswarm
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_explain, feature_names=features, show=True)

# Global feature importance — bar chart
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_explain, feature_names=features, plot_type='bar', show=True)

In [ ]:
# Local explanation — waterfall for a single prediction
sample_id = 500
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value,
    shap_values[sample_id],
    feature_names=features,
    max_display=12
)